In [1]:
# 导入所需依赖库
import re
import os
import time
import random
import cloudscraper
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup

# 配置基础URL信息
BASE_URL = 'https://awoiaf.westeros.org/index.php/List_of_characters'
DOMAIN = 'https://awoiaf.westeros.org'

# 初始化云爬虫并请求角色列表页面
scraper = cloudscraper.create_scraper()
response = scraper.get(BASE_URL)

# 检查请求状态并解析页面
page_html = ""
if response.status_code == 200:
    page_html = response.text
    soup = BeautifulSoup(page_html, 'html5lib')
print(f"角色列表页请求状态码: {response.status_code}")

# 提取页面中所有有效角色名称和对应链接
char_names = []  # 存储角色名称
char_urls = []   # 存储角色详情页链接

# 遍历页面所有<li>标签，筛选包含有效<a>标签的项
for li_tag in soup.find_all('li'):
    a_tag = li_tag.find('a')
    # 仅保留同时包含title和href属性的<a>标签
    if a_tag and set(a_tag.attrs.keys()) == {'title', 'href'}:
        char_title = a_tag['title']
        # 过滤掉包含冒号的无效标题（排除非角色类条目）
        if ':' not in char_title:
            char_names.append(char_title)
            char_urls.append(DOMAIN + a_tag['href'])

# 构建角色信息DataFrame
char_df = pd.DataFrame({
    'character_name': char_names,
    'character_url': char_urls
})
print("角色信息DataFrame预览：")
print(char_df.head())
print(f"\n共识别出潜在角色数量: {len(char_df)}")

# -------------------------- 单角色别名提取示例 --------------------------
# 以艾莉亚·史塔克为例，演示如何从详情页提取角色别名
demo_char_url = "https://awoiaf.westeros.org/index.php/Arya_Stark"
demo_response = scraper.get(demo_char_url)
aliases_list = []
if demo_response.status_code == 200:
    demo_soup = BeautifulSoup(demo_response.text, 'html5lib')
    # 遍历<th>标签，定位"Aliases"行并提取别名
    for th_tag in demo_soup.find_all('th'):
        if th_tag.text.strip() == 'Aliases':
            alias_text = th_tag.find_next_sibling('td').text.strip()
            # 移除引用标记（如[1][2]）并拆分别名
            alias_text = [name.strip() for name in re.split(r'(?:\[\d+\])+', alias_text) if name.strip()]
            aliases_list.append(alias_text)
print(f"艾莉亚·史塔克的别名: {aliases_list}")

# -------------------------- 角色详情页本地镜像 --------------------------
# 创建存储HTML文件的目录（不存在则新建）
save_dir = 'HTML'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 批量下载角色详情页并保存到本地
print(f"\n开始下载角色详情页到 {save_dir} 目录...")
for idx, row in tqdm(char_df.iterrows(), total=len(char_df), desc="下载进度", unit="页"):
    char_name = str(row['character_name'])
    char_url = row['character_url']
    
    # 确保URL是完整的绝对路径
    if not char_url.startswith('http'):
        char_url = DOMAIN + char_url
    
    # 生成安全的文件名（替换空格/斜杠避免路径错误）
    file_name = f"{save_dir}/{char_name.replace(' ', '_').replace('/', '-')}.html"
    
    # 跳过已存在的文件，避免重复下载
    if os.path.exists(file_name):
        continue
    
    try:
        # 发起请求（超时时间30秒）
        res = scraper.get(char_url, timeout=30)
        if res.status_code == 200:
            # 保存页面内容到本地
            with open(file_name, 'w', encoding='utf-8') as f:
                f.write(res.text)
        elif res.status_code == 404:
            print(f"\n[警告] 404未找到: {char_url}")
        elif res.status_code == 403:
            print(f"\n[错误] 403禁止访问: {char_url}（可能因访问过快被限制）")
            break  # 触发403时立即停止，避免被封禁
        
        # 随机延迟（1-3秒），遵守网站爬取礼仪
        time.sleep(random.uniform(1, 3))
    
    except Exception as e:
        print(f"\n下载失败 {char_name} ({char_url}): {str(e)}")

# -------------------------- 本地文件完整性校验 --------------------------
print(f"\n开始校验 {save_dir} 目录下HTML文件完整性...")
# 1. 收集文件列表用于对比
# 本地实际存在的文件
local_files = {f for f in os.listdir(save_dir) if f.endswith('.html')}
# 根据DataFrame预期应存在的文件
expected_files = {
    f"{name.replace(' ', '_').replace('/', '-')}.html" 
    for name in char_df['character_name']
}

# 2. 计算文件差异
missing_files = sorted(list(expected_files - local_files))    # 应存在但缺失的文件
extra_files = sorted(list(local_files - expected_files))     # 本地存在但非预期的文件

# 3. 检查文件完整性（空文件/截断文件）
empty_files = []       # 空文件
truncated_files = []   # 未完整下载的文件（无</html>结束标签）

for file in tqdm(local_files, desc="校验文件完整性", unit="文件"):
    file_path = os.path.join(save_dir, file)
    try:
        file_size = os.path.getsize(file_path)
        # 检查空文件
        if file_size == 0:
            empty_files.append(file)
            continue
        # 检查文件是否截断（读取最后100字节判断是否有结束标签）
        with open(file_path, 'r', encoding='utf-8') as f:
            if file_size > 100:
                f.seek(file_size - 100)
            else:
                f.seek(0)
            last_content = f.read()
            if '</html>' not in last_content.lower():
                truncated_files.append(file)
    except Exception as e:
        tqdm.write(f"读取文件失败 {file}: {str(e)}")

# -------------------------- 整合内容并保存到HTML文件 --------------------------
# 构建完整HTML内容（整合角色列表+审计报告）
final_html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>冰与火之歌角色数据</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        h1, h2, h3 {{ color: #333; }}
        table {{ border-collapse: collapse; width: 100%; margin: 20px 0; }}
        th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
        th {{ background-color: #f5f5f5; }}
        .section {{ margin: 30px 0; }}
    </style>
</head>
<body>
    <div class="section">
        <h1>冰与火之歌角色列表</h1>
        {char_df.to_html(index=False, escape=False)}
    </div>
    
    <div class="section">
        <h1>角色页面下载审计报告</h1>
        <hr>
        <h2>1. 数量对比</h2>
        <ul>
            <li>预期文件数（来自角色列表）: {len(expected_files)}</li>
            <li>本地实际文件数: {len(local_files)}</li>
            <li>匹配文件数: {len(expected_files & local_files)}</li>
            <li>缺失文件数: {len(missing_files)}</li>
            <li>额外文件数: {len(extra_files)}</li>
        </ul>
        <h2>2. 完整性问题</h2>
        <ul>
            <li>空文件数: {len(empty_files)}</li>
            <li>截断文件数: {len(truncated_files)}</li>
        </ul>
"""

# 补充缺失/额外/损坏文件详情
if missing_files:
    final_html_content += f"<h3>缺失文件（前5个）:</h3><ul>"
    for f in missing_files[:5]:
        final_html_content += f"<li>{f}</li>"
    if len(missing_files) > 5:
        final_html_content += f"<li>... 还有 {len(missing_files)-5} 个更多</li>"
    final_html_content += "</ul>"

if extra_files:
    final_html_content += f"<h3>额外文件（前5个）:</h3><ul>"
    for f in extra_files[:5]:
        final_html_content += f"<li>{f}</li>"
    if len(extra_files) > 5:
        final_html_content += f"<li>... 还有 {len(extra_files)-5} 个更多</li>"
    final_html_content += "</ul>"

if empty_files or truncated_files:
    final_html_content += f"<h3>损坏文件（前5个）:</h3><ul>"
    for f in (empty_files + truncated_files)[:5]:
        final_html_content += f"<li>{f}</li>"
    if len(empty_files + truncated_files) > 5:
        final_html_content += f"<li>... 还有 {len(empty_files + truncated_files)-5} 个更多</li>"
    final_html_content += "</ul>"

# 闭合HTML标签
final_html_content += """
    </div>
</body>
</html>
"""



print("\n✅ 所有操作完成！")



角色列表页请求状态码: 200
角色信息DataFrame预览：
        character_name                                      character_url
0        A certain man  https://awoiaf.westeros.org/index.php/A_certai...
1     Abelar Hightower  https://awoiaf.westeros.org/index.php/Abelar_H...
2               Abelon       https://awoiaf.westeros.org/index.php/Abelon
3  Addam of Duskendale  https://awoiaf.westeros.org/index.php/Addam_of...
4           Addam Frey   https://awoiaf.westeros.org/index.php/Addam_Frey

共识别出潜在角色数量: 3610
艾莉亚·史塔克的别名: [['Arya Horseface', 'Arya Underfoot', 'Arry', 'Lumpyhead', 'Lumpyface', 'Stickboy', 'Rabbitkiller', 'Weasel', 'The ghost in Harrenhal', 'Nymeria', 'Nan', 'Squab', 'Squirrel', 'Wolf girl', 'Salty', 'Cat of the Canals', 'Blind Beth', 'The blind girl', 'The night wolf', 'The ugly girl', 'Mercedene', 'Mercy']]

开始下载角色详情页到 HTML1 目录...


下载进度:   1%|▌                                                                   | 27/3610 [00:57<2:07:13,  2.13s/页]


KeyboardInterrupt: 